# Back-to-Back Causal Conv1D Forward

This notebook computes a **fused back-to-back causal conv1d** forward pass using cuDNN.

The B2B kernel fuses three stages into a single kernel launch:

1. **Projection conv** — depthwise causal conv1d across all `3*dim` channels
2. **Gating** — element-wise multiply of two projected channels, then mixer conv + skip connection
3. **Post-gating** — multiply the mixer output with the remaining projected channel

```python
proj    = causal_conv1d(x, weights_proj)          # all 3*dim channels
gated   = proj[:, 1::3, :] * proj[:, 2::3, :]     # gate: ch1 * ch2
y       = causal_conv1d(gated, weights_mixer) + skip_bias * gated  # intermediate
y_gated = y * proj[:, 0::3, :]                    # final post-gated output
```

**Tensor shapes (cuhyena convention):**

| Tensor | Shape | Description |
|---|---|---|
| x | `(batch, 3*dim, seq_len)` | 3 input channels interleaved per dim |
| weights_proj | `(3*dim, K_proj)` | projection filters |
| weights_mixer | `(dim, K_mixer)` | mixer filter |
| skip_bias | `(dim,)` | skip-connection bias |
| y_gated | `(batch, dim, seq_len)` | final post-gated output (return value) |

## Prerequisites and Setup
This notebook requires an NVIDIA GPU (Hopper or later recommended) and cuDNN 9.24.0 or later.

**Environment setup** — make sure the cuDNN runtime library and the `cudnn` Python package are discoverable before launching the notebook:

- **Option A – pip install:**
  ```bash
  pip install nvidia-cudnn-frontend
  ```
- **Option B – set paths manually:**
  ```bash
  export LD_LIBRARY_PATH=/path/to/cudnn/lib:${LD_LIBRARY_PATH}
  export PYTHONPATH=/path/to/cudnn_frontend/build_python:${PYTHONPATH}
  ```

Adjust the paths above to match your local build or installation directory.

In [1]:
# !nvidia-smi

In [2]:
# !pip install nvidia-cudnn-cu12
# !pip install nvidia-cudnn-frontend
# !pip3 install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu128

## Overview

We will perform a fused B2B causal conv1d forward pass with:

- batch size: 2
- dim (channels): 64
- sequence length: 512
- projection kernel size: 4
- mixer kernel size: 7
- data type: Float32

We compare the cuDNN fused result against a decomposed PyTorch reference.

In [3]:
import torch
import torch.nn.functional as F
import cudnn

print("cuDNN backend version:", cudnn.backend_version())

cuDNN backend version: 92300


In [4]:
def causal_conv1d_ref(x, weight):
    """Depthwise causal conv1d.

    Args:
        x:      (batch, channels, seq_len)
        weight: (channels, kernel_size)
    """
    x_padded = F.pad(x, (weight.shape[1] - 1, 0))
    return F.conv1d(x_padded, weight.unsqueeze(1), groups=x.shape[1])


def b2b_causal_conv1d_ref(x, weights_proj, weights_mixer, skip_bias):
    """Reference: decomposed B2B causal conv1d.

    Returns the post-gated final output ``y_gated``.

    Args:
        x:             (batch, 3*dim, seq_len)
        weights_proj:  (3*dim, K_proj)
        weights_mixer: (dim, K_mixer)
        skip_bias:     (dim,)
    """
    proj = causal_conv1d_ref(x, weights_proj)
    gated = proj[:, 1::3, :] * proj[:, 2::3, :]
    y = causal_conv1d_ref(gated, weights_mixer) + skip_bias[:, None] * gated
    y_gated = y * proj[:, 0::3, :]
    return y_gated

In [5]:
batch = 2
dim = 64
seq_len = 512
k_proj = 4
k_mixer = 7
dtype = torch.float32

has_cuda = torch.cuda.is_available()

torch.manual_seed(42)

x = torch.randn(batch, 3 * dim, seq_len, dtype=dtype)
weights_proj = torch.randn(3 * dim, k_proj, dtype=dtype)
weights_mixer = torch.randn(dim, k_mixer, dtype=dtype)
skip_bias = torch.randn(dim, dtype=dtype)

print(f"CUDA available: {has_cuda}")
print(f"x:             {x.shape}, dtype={x.dtype}")
print(f"weights_proj:  {weights_proj.shape}, dtype={weights_proj.dtype}")
print(f"weights_mixer: {weights_mixer.shape}, dtype={weights_mixer.dtype}")
print(f"skip_bias:     {skip_bias.shape}, dtype={skip_bias.dtype}")

CUDA available: True
x:             torch.Size([2, 192, 512]), dtype=torch.float32
weights_proj:  torch.Size([192, 4]), dtype=torch.float32
weights_mixer: torch.Size([64, 7]), dtype=torch.float32
skip_bias:     torch.Size([64]), dtype=torch.float32


In [6]:
y_gated_ref = b2b_causal_conv1d_ref(x, weights_proj, weights_mixer, skip_bias)
print(f"y_gated_ref: {y_gated_ref.shape}, dtype={y_gated_ref.dtype}")

y_gated_ref: torch.Size([2, 64, 512]), dtype=torch.float32


In [7]:
if has_cuda:
    y_gated_cudnn = cudnn.ops.b2b_causal_conv1d(
        x.cuda(), weights_proj.cuda(), weights_mixer.cuda(), skip_bias.cuda(),
    )
    print(f"y_gated_cudnn: {y_gated_cudnn.shape}, dtype={y_gated_cudnn.dtype}")
else:
    print("Skipping cuDNN forward (no CUDA device).")

y_gated_cudnn: torch.Size([2, 64, 512]), dtype=torch.float32


In [8]:
if has_cuda:
    atol = 1e-3

    max_abs = (y_gated_cudnn.cpu().float() - y_gated_ref.float()).abs().max().item()
    print(f"y_gated max_abs_diff={max_abs:.4e}")
    assert max_abs < atol, f"y_gated verification failed: max_abs={max_abs}"

    print("\nPASSED: cuDNN B2B causal_conv1d forward matches reference.")
else:
    print("Skipping verification (no CUDA device).")

y_gated max_abs_diff=1.5259e-04

PASSED: cuDNN B2B causal_conv1d forward matches reference.
